In [1]:
import os
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

from pathlib import Path
import cv2
import pickle
import numpy as np
import matplotlib.pyplot as plt
import math
import json
from tqdm import tqdm
from datetime import date
plt.rcParams["figure.figsize"] = (24,18)

from ultralytics import YOLO
from cv_utils import *
from cv_pipeline import *

In [2]:
# YOLO model path
model_path = os.path.join(os.path.dirname(os.getcwd()), 'models')
pitch_segment_model = smp.Unet("resnet34", encoder_weights="imagenet", activation=None, classes=7)
pitch_segment_model.load_state_dict(torch.load(os.path.join(model_path, "res_unet_512.pth")))

player_detect_model = YOLO(os.path.join(model_path, "yolov8n_2nd_train.pt"))

# General data path
data_path = os.path.join(os.path.dirname(os.getcwd()), 'data')

#### Crop images to football pitch

In [17]:
all_cropped_destination = []
for vid_name in ['real_train', 'synth_train']:
    print(vid_name)
    image_root_path = os.path.join(data_path, 'images/' + vid_name)
    destination_path = os.path.join(data_path, 'images/cropped_' + vid_name)
    if not os.path.exists(destination_path):
        os.mkdir(destination_path)
    for image_name in tqdm(os.listdir(image_root_path)):
        image_path = os.path.join(image_root_path, image_name)
        image = cv2.imread(image_path, cv2.COLOR_BGR2RGB)
        orig_w, orig_h = image.shape[:2]

        # Crop the image to only the pitch
        predicted_mask, colored_mask = pitch_segment(image, pitch_segment_model)
        pitch_area = cv2.inRange(predicted_mask, 41, 255)
        _, rect, _ = detect_largest_contour(pitch_area)

        startX, startY, w, h = rect
        endX, endY = startX + w, startY + h
        cropped_image = cv2.resize(image[startY:endY, startX:endX], (orig_h, orig_w))

        # Write image to destination
        destination_image_path = os.path.join(destination_path, image_name)
        cv2.imwrite(destination_image_path, cropped_image)
    print(destination_path)
    all_cropped_destination.append(destination_path)

real_train


100%|██████████| 457/457 [01:45<00:00,  4.35it/s]


/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train
synth_train


100%|██████████| 159/159 [00:53<00:00,  2.96it/s]

/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train


#### Generate COCO dataset

In [14]:
# Generate one or multiple COCO datasets
destination_path_list = []
for folder_path in all_cropped_destination:
    vid_name = os.path.basename(folder_path)
    print(vid_name)
    print(os.path.join(data_path, 'images' + '/' + vid_name))
    dest_coco_path = export_coco_dataset_from_prediction(data_path, vid_name, 
                                        model_path=model_path, model_name="yolov8n_2nd_train.pt")
    
    destination_path_list.append(dest_coco_path)

cropped_real_train
/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train


  0%|          | 0/457 [00:00<?, ?it/s]


image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000144.png: 576x1024 18 persons, 3.0ms
Speed: 2.1ms preprocess, 3.0ms inference, 0.7ms postprocess per image at shape (1, 3, 576, 1024)
  0%|          | 1/457 [00:00<00:52,  8.71it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000186.png: 576x1024 17 persons, 3.0ms
Speed: 2.0ms preprocess, 3.0ms inference, 0.8ms postprocess per image at shape (1, 3, 576, 1024)

image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000187.png: 576x1024 17 persons, 3.3ms
Speed: 2.2ms preprocess, 3.3ms inference, 1.1ms postprocess per image at shape (1, 3, 576, 1024)
  1%|          | 3/457 [00:00<00:40, 11.24it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_real_train/000000000204.png: 576x1024 20 person

cropped_synth_train
/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train


  0%|          | 0/159 [00:00<?, ?it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train/000000000005.png: 576x1024 23 persons, 3.4ms
Speed: 2.1ms preprocess, 3.4ms inference, 0.7ms postprocess per image at shape (1, 3, 576, 1024)
  1%|          | 1/159 [00:00<00:24,  6.37it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train/000000000006.png: 576x1024 24 persons, 2.9ms
Speed: 2.2ms preprocess, 2.9ms inference, 0.7ms postprocess per image at shape (1, 3, 576, 1024)
  1%|▏         | 2/159 [00:00<00:20,  7.50it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/images/cropped_synth_train/000000000009.png: 576x1024 25 persons, 3.6ms
Speed: 2.6ms preprocess, 3.6ms inference, 1.0ms postprocess per image at shape (1, 3, 576, 1024)
  2%|▏         | 3/159 [00:00<00:20,  7.44it/s]
image 1/1 /media/khoa-ys/Personal/Projects/Football Analy

#### Merge multiple COCO datasets

In [3]:
coco_dataset_path_1 = os.path.join(data_path, 'coco_datasets/cropped_synth_train_fixed')
coco_dataset_path_2 = os.path.join(data_path, 'coco_datasets/cropped_real_train_fixed')

dest_path=os.path.join(data_path, 'coco_datasets')

# Merge generated COCO dataset to one dataset for model training
merge_dest_path = merge_coco_dataset(coco_dataset_path_1, coco_dataset_path_2,
                   dest_path=os.path.join(data_path, 'coco_datasets'))

#### Convert COCO dataset to YOLO format

In [4]:
# Load COCO dataset
dataset_name = 'cropped_real_synth_train_fixed'
coco_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
test_dataset_path = os.path.join(data_path, 'coco_datasets/' + 'real_test_fixed')
# Convert COCO to YOLO
coco2yolo(coco_dataset_path, test_dataset_path=test_dataset_path)

  0%|          | 0/492 [00:00<?, ?it/s]

100%|██████████| 300/300 [00:25<00:00, 11.80it/s]


#### Augment YOLO dataset

In [5]:
dataset_name = 'cropped_real_synth_train_fixed_yolov8'
yolo_dataset_path = os.path.join(data_path, 'coco_datasets/' + dataset_name)
yolo_metadata_path = os.path.join(yolo_dataset_path, 'data.yaml')

augment_yolo(yolo_metadata_path, yolo_dataset_path)

/media/khoa-ys/Personal/Projects/Football Analysis/Football-analysis/data/coco_datasets/cropped_real_synth_train_fixed_yolov8/train/images


100%|██████████| 492/492 [01:41<00:00,  4.87it/s]

4428 4428
